# Step 3b: Quality Analysis & Threshold Selection

This notebook loads the scored manifest and helps you:
1. Visualize DNSMOS score distributions per dataset
2. Sweep thresholds and see how many hours survive
3. Pick the right quality/volume trade-off
4. Listen to samples near the threshold boundary

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.manifest import read_manifest

SCORED_MANIFEST = '../data/scored/manifest.jsonl'

entries = read_manifest(SCORED_MANIFEST)
df = pd.DataFrame(entries)
print(f'Total samples: {len(df)}')
print(f'Total hours: {df["duration_s"].sum() / 3600:.1f}')
print(f'Datasets: {df["source_dataset"].value_counts().to_dict()}')
df.head()

## 1. DNSMOS Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['dnsmos_ovrl', 'dnsmos_sig', 'dnsmos_bak']):
    for ds_name, group in df.groupby('source_dataset'):
        ax.hist(group[metric], bins=50, alpha=0.6, label=ds_name, density=True)
    ax.set_xlabel(metric)
    ax.set_ylabel('Density')
    ax.set_title(f'{metric} distribution')
    ax.legend()

plt.tight_layout()
plt.show()

## 2. Per-Dataset Summary Statistics

In [ ]:
summary = df.groupby('source_dataset').agg(
    count=('utt_id', 'count'),
    hours=('duration_s', lambda x: x.sum() / 3600),
    ovrl_mean=('dnsmos_ovrl', 'mean'),
    ovrl_std=('dnsmos_ovrl', 'std'),
    ovrl_p10=('dnsmos_ovrl', lambda x: np.percentile(x, 10)),
    ovrl_p50=('dnsmos_ovrl', 'median'),
    bak_mean=('dnsmos_bak', 'mean'),
    sig_mean=('dnsmos_sig', 'mean'),
).round(2)
summary

## 3. Surviving Hours vs. DNSMOS OVRL Threshold

This is the key plot for picking your threshold.

In [ ]:
thresholds = np.arange(2.0, 4.5, 0.25)

fig, ax = plt.subplots(figsize=(10, 6))

# Per-dataset curves
for ds_name, group in df.groupby('source_dataset'):
    hours = [group[group['dnsmos_ovrl'] >= t]['duration_s'].sum() / 3600 for t in thresholds]
    ax.plot(thresholds, hours, marker='o', label=ds_name)

# Total curve
total_hours = [df[df['dnsmos_ovrl'] >= t]['duration_s'].sum() / 3600 for t in thresholds]
ax.plot(thresholds, total_hours, marker='s', linewidth=2.5, color='black', label='TOTAL')

# Target band
ax.axhspan(200, 500, alpha=0.1, color='green', label='Target: 200-500 hrs')

ax.set_xlabel('DNSMOS OVRL threshold (>=)')
ax.set_ylabel('Surviving hours')
ax.set_title('Hours remaining vs. quality threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print table
print('\nThreshold -> Total Hours:')
for t, h in zip(thresholds, total_hours):
    marker = ' <-- suggested' if abs(t - 3.0) < 0.01 else ''
    print(f'  OVRL >= {t:.2f}: {h:.1f} hrs{marker}')

## 4. Joint Threshold Sweep (OVRL + BAK)

In [ ]:
ovrl_range = np.arange(2.5, 4.25, 0.25)
bak_range = np.arange(2.0, 4.25, 0.25)

hours_grid = np.zeros((len(ovrl_range), len(bak_range)))
for i, ovrl_t in enumerate(ovrl_range):
    for j, bak_t in enumerate(bak_range):
        mask = (df['dnsmos_ovrl'] >= ovrl_t) & (df['dnsmos_bak'] >= bak_t)
        hours_grid[i, j] = df[mask]['duration_s'].sum() / 3600

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(hours_grid, origin='lower', aspect='auto',
               extent=[bak_range[0], bak_range[-1], ovrl_range[0], ovrl_range[-1]])
ax.set_xlabel('DNSMOS BAK threshold (>=)')
ax.set_ylabel('DNSMOS OVRL threshold (>=)')
ax.set_title('Surviving hours (joint threshold)')
plt.colorbar(im, label='Hours')

# Annotate cells
for i, ovrl_t in enumerate(ovrl_range):
    for j, bak_t in enumerate(bak_range):
        ax.text(bak_t, ovrl_t, f'{hours_grid[i,j]:.0f}',
                ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

## 5. Listen to Boundary Samples

Sample audio near the threshold to sanity-check your choice.

In [ ]:
from IPython.display import Audio, display

CHOSEN_THRESHOLD = 3.0  # <-- adjust after looking at the plots above

boundary = df[
    (df['dnsmos_ovrl'] >= CHOSEN_THRESHOLD - 0.2) &
    (df['dnsmos_ovrl'] <= CHOSEN_THRESHOLD + 0.2)
].sample(n=min(5, len(df)), random_state=42)

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

for _, row in boundary.iterrows():
    print(f"\n{row['utt_id']} | {row['source_dataset']} | "
          f"OVRL={row['dnsmos_ovrl']:.2f} SIG={row['dnsmos_sig']:.2f} BAK={row['dnsmos_bak']:.2f} | "
          f"{row['duration_s']:.1f}s")
    print(f"  Transcript: {row.get('transcript', 'N/A')}")
    audio_path = os.path.join(BASE_DIR, row['audio_path'])
    display(Audio(filename=audio_path))

## 6. Export Chosen Thresholds

Once you've decided, update `config.yaml` with your chosen values, then run:
```bash
python filter_dedup.py --config config.yaml
```

In [ ]:
# Final summary with your chosen threshold
OVRL_THRESH = 3.0   # <-- set your final choice
BAK_THRESH = 2.5    # <-- set your final choice (or None to skip)

mask = df['dnsmos_ovrl'] >= OVRL_THRESH
if BAK_THRESH is not None:
    mask = mask & (df['dnsmos_bak'] >= BAK_THRESH)

filtered = df[mask]
print(f'Thresholds: OVRL >= {OVRL_THRESH}, BAK >= {BAK_THRESH}')
print(f'Surviving samples: {len(filtered)}')
print(f'Surviving hours: {filtered["duration_s"].sum() / 3600:.1f}')
print(f'\nPer dataset:')
for ds, group in filtered.groupby('source_dataset'):
    print(f'  {ds}: {len(group)} samples, {group["duration_s"].sum()/3600:.1f} hrs')